In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# Find project root automatically
# --------------------------------------------------

current = Path.cwd().resolve()

# Walk upward until we find the folder containing "src"
while not (current / "src").exists():
    if current.parent == current:
        raise FileNotFoundError("Could not locate project root containing 'src'.")
    current = current.parent

PROJECT_ROOT = current
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# --------------------------------------------------
# Imports
# --------------------------------------------------

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)

from agents.baseline_agents import (
    FixedPriceAgent,
    RandomAgent,
    TimedPricingAgent,
    DemandBasedAgent,
    LinearDecayAgent
)

from utils.evaluator import (
    evaluate_agent,
    compare_agents,
    plot_agent_comparison
)

plt.style.use("seaborn-v0_8")

print("✅ All modules loaded!")
print(f"Project Root : {PROJECT_ROOT}")
print(f"Source Path  : {SRC_PATH}")
print(f"Price Levels : {PRICE_LEVELS}")

In [ ]:
env = DynamicPricingEnv(
    max_inventory=50,
    max_days=30
)

agents = [
    FixedPriceAgent(env, price_idx=2),
    RandomAgent(env),
    TimedPricingAgent(env),
    DemandBasedAgent(env),
    LinearDecayAgent(env)
]

print("=== AGENTS CREATED ===")
for agent in agents:
    print(f"  → {agent.name}")

In [ ]:
print("Running 100 episodes per agent...")
print("This takes about 1-2 minutes...\n")

all_results, summary_df = compare_agents(
    agents,
    n_episodes=100
)

In [ ]:
plot_agent_comparison(
    all_results,
    summary_df,
    save_path='../results/baseline_comparison.png'
)

In [ ]:
# Show price trajectory for each agent
fig, axes = plt.subplots(
    2, 3, figsize=(16, 10)
)
axes = axes.flatten()
colors = ['steelblue', 'coral', 'green',
          'purple', 'orange']

for i, agent in enumerate(agents):
    result = agent.run_episode(seed=42)
    prices = result['prices_used']

    axes[i].plot(
        prices,
        color=colors[i],
        linewidth=2,
        marker='o',
        markersize=3
    )
    axes[i].set_title(
        f'{agent.name}\n'
        f'Revenue: ${result["total_revenue"]:.0f}',
        fontweight='bold'
    )
    axes[i].set_xlabel('Day')
    axes[i].set_ylabel('Price ($)')
    axes[i].set_ylim([0, 350])
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(
        y=150, color='red',
        linestyle='--',
        alpha=0.5,
        label='$150 baseline'
    )

axes[-1].axis('off')
plt.suptitle(
    'Price Trajectories — Baseline Agents\n'
    '(RL agent must beat these!)',
    fontsize=14,
    fontweight='bold'
)
import os

plt.tight_layout()

# ----------------------------------------
# Create results folder automatically
# ----------------------------------------

PROJECT_ROOT = os.getcwd()

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "baseline_trajectories.png"
)

plt.savefig(
    SAVE_PATH,
    bbox_inches="tight",
    dpi=150
)

plt.show()

print(f"✅ Trajectories saved!")
print(f"Saved at: {SAVE_PATH}")

In [ ]:
# Best baseline = target for RL agent to beat
best_baseline = summary_df.iloc[0]

print("╔══════════════════════════════════════════╗")
print("║    WEEK 1 DAY 2 — BASELINE SUMMARY       ║")
print("╠══════════════════════════════════════════╣")
print("║  AGENTS TESTED:                          ║")
for _, row in summary_df.iterrows():
    print(f"║  → {row['Agent']:<20}: "
          f"${row['Mean Revenue']:<10.0f}       ║")
print("╠══════════════════════════════════════════╣")
print(f"║  BEST BASELINE: {best_baseline['Agent']:<26} ║")
print(f"║  Revenue      : "
      f"${best_baseline['Mean Revenue']:.0f}"
      f"{'':<23} ║")
print("╠══════════════════════════════════════════╣")
print("║  RL TARGET: Beat best baseline! 🎯       ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Q-Learning! 🧠               ║")
print("╚══════════════════════════════════════════╝")